# Phase 2 unified Drive smoke test
Read-only: consumes existing IBKR, TXF and Taiwan FINAL GATE artifacts. It does not rerun historical QA or write Drive data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install 'git+https://github.com/hh4832/market-research-core.git@feature/phase2-unified-data-backtest'

In [ ]:
from pathlib import Path
from market_research_core import BacktestAssumptions, BacktestEngine, MarketDataClient, MarketDataConfig, UnifiedBars

drive_root = Path('/content/drive/MyDrive')
def unique_dir(name):
    hits = [p for p in drive_root.rglob(name) if p.is_dir()]
    if len(hits) != 1: raise RuntimeError(f'Expected one {name}; found {len(hits)}')
    return hits[0]
ibkr_dir = unique_dir('20261101_205431_06359faf7ef4_IBKR_FINAL_GATE_FULL')
txf_dir = unique_dir('20261014_213932_404db609cf13')
lake_marker = unique_dir('_tw-market-data-lake')
data_root = lake_marker.parent
client = MarketDataClient(MarketDataConfig(ibkr_run_dir=ibkr_dir, txf_run_dir=txf_dir, taiwan_lake_root=data_root))
print('Pinned roots:', ibkr_dir, txf_dir, data_root, sep='\n')

In [ ]:
ibkr = client.load(market='IBKR', symbol='TSM')
txf = client.load(market='TXF', symbol='TXF')
tw = client.load(market='TW', symbol='2330', usage='BACKTEST')
for name, loaded in [('IBKR', ibkr), ('TXF', txf), ('TW', tw)]:
    assert len(loaded.frame) > 20
    assert {'data_exists','research_eligible','backtest_eligible','execution_eligible','available_at'} <= set(loaded.frame.columns)
    probe = UnifiedBars(loaded.frame.tail(2000).copy(), loaded.provenance, loaded.native_metadata)
    result = BacktestEngine().run_sma(probe, BacktestAssumptions(fast_window=5, slow_window=20))
    assert result.provenance['performance_status'] == 'RESEARCH_INTEGRATION_DEMO'
    print(name, 'PASS', 'rows=', len(loaded.frame), 'trades=', result.trade_count, 'snapshot=', loaded.provenance.snapshot_id)
print('PHASE 2 UNIFIED DRIVE SMOKE: PASS')